In [1]:
%cd NISP-Dataset/Hindi_master

/data/himanshu/NISP-Dataset/Hindi_master


/data/himanshu/new_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
dataset_path = '/data/himanshu/NISP-Dataset/Hindi_master'  # Update this path as needed
os.chdir(dataset_path)

In [3]:
import pandas as pd

# Define the path to the speaker info file (adjust if necessary)
spkr_info_path = '../total_spkrinfo.list'
spkr_columns = ['Speaker_ID', 'Gender', 'Mother_Tongue', 'Height_cm',
                'Shoulder_size_cm', 'Waist_size_cm', 'Weight_kg', 'Age_y',
                'Native_State', 'Native_District']

# Load the speaker info into a DataFrame
spkr_info_df = pd.read_csv(spkr_info_path, sep=' ', names=spkr_columns)
spkr_info_df = spkr_info_df.drop(index=0)
print("Speaker Metadata (first 5 rows):")
print(spkr_info_df.head())

Speaker Metadata (first 5 rows):
  Speaker_ID  Gender Mother_Tongue Height_cm Shoulder_size_cm Waist_size_cm  \
1   Hin_0001  Female         Hindi       163               40          89.5   
2   Hin_0002  Female         Hindi     154.5             36.5            72   
3   Hin_0003    Male         Hindi     167.5             40.5            78   
4   Hin_0004    Male         Hindi       176               43          91.5   
5   Hin_0005  Female         Hindi       153             40.5            96   

  Weight_kg  Age_y    Native_State Native_District  
1      58.5  24.24       Rajasthan          Jaipur  
2      50.9  26.06  Madhya_Pradesh          Indore  
3      56.6  21.51         Haryana       Faridabad  
4      77.6  21.09    Chhattisgarh        Bilaspur  
5      80.2  27.39   Uttar_Pradesh    Kanpur_Nagar  


In [4]:
import os

In [5]:
import glob

In [6]:
# Define the folder containing merged audio files
audio_folder = "../Merged_Audio"

# List all .wav files in the merged folder
audio_files = glob.glob(os.path.join(audio_folder, "*.wav"))

# Build a list to store audio file information
audio_data = []
for file_path in audio_files:
    file_name = os.path.basename(file_path)
    # Expected filename format: Hin_0067_Hin_m_0014.wav
    parts = file_name.split('_')
    if len(parts) >= 2:
        # Construct the full speaker ID from the first two tokens, e.g., "Hin_0067"
        speaker_id_full = f"{parts[0]}_{parts[1]}"

        audio_data.append({
            'Speaker_ID': speaker_id_full,

            'File_Path': file_path
        })
    else:
        print("Unexpected filename format:", file_name)

# Create a DataFrame from the audio data
audio_df = pd.DataFrame(audio_data)
print("Audio DataFrame (first 5 rows):")
print(audio_df.head())

Audio DataFrame (first 5 rows):
  Speaker_ID                                File_Path
0   Hin_0092  ../Merged_Audio/Hin_0092_Hin_f_0008.wav
1   Hin_0077  ../Merged_Audio/Hin_0077_Eng_f_7341.wav
2   Hin_0093  ../Merged_Audio/Hin_0093_Hin_f_0023.wav
3   Hin_0020  ../Merged_Audio/Hin_0020_Eng_m_5447.wav
4   Hin_0013  ../Merged_Audio/Hin_0013_Eng_m_5222.wav


In [7]:
audio_df['Speaker_ID'] = audio_df['Speaker_ID'].astype(str)
spkr_info_df['Speaker_ID'] = spkr_info_df['Speaker_ID'].astype(str)


merged_df = pd.merge(audio_df, spkr_info_df[['Speaker_ID', 'Native_State']],
                     on='Speaker_ID', how='left')

# Select only the desired columns: the audio file path and the corresponding Native_State.
final_df = merged_df[['File_Path', 'Speaker_ID', 'Native_State']]
print("Final Dataset (first 5 rows):")
print(final_df.head())

Final Dataset (first 5 rows):
                                 File_Path Speaker_ID   Native_State
0  ../Merged_Audio/Hin_0092_Hin_f_0008.wav   Hin_0092  Uttar_Pradesh
1  ../Merged_Audio/Hin_0077_Eng_f_7341.wav   Hin_0077    Maharashtra
2  ../Merged_Audio/Hin_0093_Hin_f_0023.wav   Hin_0093    Maharashtra
3  ../Merged_Audio/Hin_0020_Eng_m_5447.wav   Hin_0020  Uttar_Pradesh
4  ../Merged_Audio/Hin_0013_Eng_m_5222.wav   Hin_0013          Bihar


In [8]:
df= final_df
df.head()

,File_Path,Speaker_ID,Native_State
0,../Merged_Audio/Hin_0092_Hin_f_0008.wav,Hin_0092,Uttar_Pradesh
1,../Merged_Audio/Hin_0077_Eng_f_7341.wav,Hin_0077,Maharashtra
2,../Merged_Audio/Hin_0093_Hin_f_0023.wav,Hin_0093,Maharashtra
3,../Merged_Audio/Hin_0020_Eng_m_5447.wav,Hin_0020,Uttar_Pradesh
4,../Merged_Audio/Hin_0013_Eng_m_5222.wav,Hin_0013,Bihar


In [9]:
import os
import pandas as pd
import random
import librosa
import numpy as np
import soundfile as sf
from collections import Counter
from imblearn.over_sampling import RandomOverSampler

In [10]:
state_counts = df["Native_State"].value_counts()
print("Original Class Distribution:\n", state_counts)

Original Class Distribution:
 Native_State
Uttar_Pradesh       1579
Rajasthan            971
Madhya_Pradesh       926
Delhi                788
Maharashtra          749
Bihar                626
Uttarakhand          320
Chhattisgarh         306
Haryana              294
Karnataka            220
Jharkhand            155
West_Bengal          154
Gujarat              151
Jammu_&_Kashmir      150
Odisha               150
Panjab                76
Himachal_Pradesh      76
Andhra_Pradesh        75
Meghalaya             73
Name: count, dtype: int64


In [11]:
# Compute Mean
mean_value = np.mean(state_counts)

# Compute Median
median_value = np.median(state_counts)

print(f"Mean Class Size: {mean_value:.2f}")
print(f"Median Class Size: {median_value}")

Mean Class Size: 412.58
Median Class Size: 220.0


In [12]:
# Define balancing thresholds
TARGET_COUNT = 350  # Set target samples per state

In [13]:
# Step 1: Undersampling Majority Classes
def undersample(df, target_count):
    df_majority = df[df["Native_State"].map(state_counts) > target_count]
    df_minority = df[df["Native_State"].map(state_counts) <= target_count]
    
    # Randomly sample from majority class
    df_majority_sampled = df_majority.groupby("Native_State").apply(lambda x: x.sample(n=target_count, random_state=42))
    
    return pd.concat([df_majority_sampled, df_minority])

In [14]:
df_balanced = undersample(df, TARGET_COUNT)
print("After Undersampling:\n", df_balanced["Native_State"].value_counts())

After Undersampling:
 Native_State
Bihar               350
Delhi               350
Madhya_Pradesh      350
Maharashtra         350
Rajasthan           350
Uttar_Pradesh       350
Uttarakhand         320
Chhattisgarh        306
Haryana             294
Karnataka           220
Jharkhand           155
West_Bengal         154
Gujarat             151
Odisha              150
Jammu_&_Kashmir     150
Himachal_Pradesh     76
Panjab               76
Andhra_Pradesh       75
Meghalaya            73
Name: count, dtype: int64


/tmp/ipykernel_161700/1930286831.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_majority_sampled = df_majority.groupby("Native_State").apply(lambda x: x.sample(n=target_count, random_state=42))


In [15]:
import librosa
import numpy as np
import soundfile as sf
import random
import os

def augment_audio(file_path, output_dir, sr=16000, augment_count=3):
    """Applies data augmentation to audio and saves new samples"""
    # Fixed the syntax error with the asterisk
    audio, _ = librosa.load(file_path, sr=sr)
    
    def apply_time_stretch(audio, rate):
        """Apply time-stretching correctly"""
        return librosa.effects.time_stretch(audio ,rate=rate)
      
    
    def apply_pitch_shift(audio, sr, n_steps):
        """Corrected pitch shift function"""
        return librosa.effects.pitch_shift(y=audio, sr=sr, n_steps=n_steps)
    
    def add_noise(audio, noise_level=0.005):
        noise = np.random.randn(len(audio)) * noise_level
        return audio + noise
    
    augmented_files = []
    for i in range(augment_count):
        augmented_audio = audio
        if random.choice([True, False]):
            augmented_audio = apply_time_stretch(augmented_audio, rate=random.uniform(0.9, 1.1))
        if random.choice([True, False]):
            augmented_audio = apply_pitch_shift(augmented_audio, sr, n_steps=random.randint(-2, 2))
        if random.choice([True, False]):
            augmented_audio = add_noise(augmented_audio)
        # Save augmented file
        file_name = os.path.basename(file_path).replace(".wav", f"_aug{i}.wav")
        output_path = os.path.join(output_dir, file_name)
        sf.write(output_path, augmented_audio, sr)
        augmented_files.append(output_path)
    
    return augmented_files

In [16]:
minority_states = df_balanced["Native_State"].value_counts()[df_balanced["Native_State"].value_counts() < TARGET_COUNT].index.tolist()
output_dir = "augmented_audio"
os.makedirs(output_dir, exist_ok=True)

augmented_data = []
for state in minority_states:
    state_df = df_balanced[df_balanced["Native_State"] == state]
    files_needed = TARGET_COUNT - len(state_df)
    
    for _, row in state_df.iterrows():
        new_files = augment_audio(row["File_Path"], output_dir, augment_count=3)  # Generate 3 per file
        for file in new_files:
            augmented_data.append({"File_Path": file, "Speaker_ID": row["Speaker_ID"], "Native_State": state})
        
        files_needed -= len(new_files)
        if files_needed <= 0:
            break

# Convert augmented data to DataFrame and merge
df_augmented = pd.DataFrame(augmented_data)
df_final = pd.concat([df_balanced, df_augmented])

print("Final Class Distribution:\n", df_final["Native_State"].value_counts())

Final Class Distribution:
 Native_State
West_Bengal         352
Karnataka           352
Gujarat             352
Odisha              351
Jammu_&_Kashmir     351
Chhattisgarh        351
Haryana             351
Madhya_Pradesh      350
Delhi               350
Bihar               350
Maharashtra         350
Jharkhand           350
Uttarakhand         350
Rajasthan           350
Uttar_Pradesh       350
Himachal_Pradesh    304
Panjab              304
Andhra_Pradesh      300
Meghalaya           292
Name: count, dtype: int64


In [48]:
df_final.describe()

,Duration
count,6460.000000
mean,9.326922
std,3.887011
min,1.781250
25%,6.562500
50%,8.406250
75%,11.143313
max,42.343750


In [49]:

# Reset the index if it's a problem
df_final = df_final.reset_index(drop=True)

# Make sure column names don't have spaces or special characters
print(df_final.columns)

# Try viewing with to_string() method for cleaner output
print(df_final.head().to_string())

# Or display as HTML in Jupyter
from IPython.display import display
display(df_final.head())

Index(['File_Path', 'Speaker_ID', 'Native_State', 'Duration'], dtype='object')
                                 File_Path Speaker_ID Native_State  Duration
0  ../Merged_Audio/Hin_0014_Hin_m_0035.wav   Hin_0014        Bihar   7.00000
1  ../Merged_Audio/Hin_0052_Eng_m_0001.wav   Hin_0052        Bihar   6.62500
2  ../Merged_Audio/Hin_0052_Eng_m_0003.wav   Hin_0052        Bihar  12.09375
3  ../Merged_Audio/Hin_0029_Hin_m_0032.wav   Hin_0029        Bihar   7.87500
4  ../Merged_Audio/Hin_0052_Eng_m_6490.wav   Hin_0052        Bihar   7.03125


,File_Path,Speaker_ID,Native_State,Duration
0,../Merged_Audio/Hin_0014_Hin_m_0035.wav,Hin_0014,Bihar,7.00000
1,../Merged_Audio/Hin_0052_Eng_m_0001.wav,Hin_0052,Bihar,6.62500
2,../Merged_Audio/Hin_0052_Eng_m_0003.wav,Hin_0052,Bihar,12.09375
3,../Merged_Audio/Hin_0029_Hin_m_0032.wav,Hin_0029,Bihar,7.87500
4,../Merged_Audio/Hin_0052_Eng_m_6490.wav,Hin_0052,Bihar,7.03125


In [50]:
df_final.shape


(6460, 4)

In [52]:
df_final['Native_State'] = pd.factorize(df_final['Native_State'])[0]

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import scipy.signal as signal
import soundfile as sf

In [ ]:
def extract_salient_segments(audio, sr, num_segments=1, segment_duration=3.0):
    """
    Extracts `num_segments` segments each of `segment_duration` seconds from the audio
    based on points of maximum spectral change.
    """
    hop_length = 512
    n_fft = 2048

    # Compute the magnitude spectrogram
    S = np.abs(librosa.stft(audio, n_fft=n_fft, hop_length=hop_length))

    # Compute spectral flux: change in the spectrum between successive frames
    flux = np.sqrt(np.sum(np.diff(S, axis=1)**2, axis=0))
    flux = np.concatenate(([0], flux))  # Align flux with frames

    # Detect peaks in the spectral flux that are above the average flux level
    peaks, _ = signal.find_peaks(flux, height=np.mean(flux))
    if len(peaks) == 0:
        print("No significant peaks found in audio.")
        return []

    # Sort the detected peaks by flux strength (largest first)
    sorted_peaks = peaks[np.argsort(flux[peaks])][::-1]

    seg_length_samples = int(segment_duration * sr)
    segments = []

    # Extract segments centered around the strongest peaks
    for peak in sorted_peaks[:num_segments]:
        peak_time = librosa.frames_to_time(peak, sr=sr, hop_length=hop_length)
        center_sample = int(peak_time * sr)
        start = max(0, center_sample - seg_length_samples // 2)
        end = start + seg_length_samples
        if end > len(audio):
            end = len(audio)
            start = max(0, end - seg_length_samples)
        segments.append(audio[start:end])

    return segments

In [ ]:
def process_dataframe(df, output_dir="extracted_segments", output_db_csv="extracted_segments_db.csv"):
    """
    For each audio file in the DataFrame:
      - If its duration is at least 8 seconds, extract 2 segments (each 4 seconds long)
        based on spectral flux.
      - Save the extracted segments as WAV files.
      - Record the extracted file paths along with the corresponding native state in a new CSV database.
    """
    # Create the output directory if it doesn't exist.
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    records = []

    # Process each row in the DataFrame.
    for index, row in df.iterrows():
        file_path = row["File_Path"]
        speaker_id = row["Speaker_ID"]
        native_state = row["Native_State"]

        if not os.path.exists(file_path):
            print(f"File not found: {file_path}")
            continue

        # Load the audio file with its original sample rate. This line was incorrectly indented.
        audio, sr = librosa.load(file_path, sr=None)

        # Get the duration of the audio in seconds.
        duration = librosa.get_duration(y=audio, sr=sr)


        # Only process if the duration is at least 8 seconds.
        if duration < 3.0:
            continue

        # Extract 2 segments, each 4 seconds long.
        segments = extract_salient_segments(audio,sr, num_segments=1, segment_duration=3.0)

        base_name = os.path.splitext(os.path.basename(file_path))[0]
        for seg_index, segment in enumerate(segments):
            # Save each segment as a WAV file.
            output_file = os.path.join(output_dir, f"{base_name}_seg{seg_index+1}.wav")
            sf.write(output_file, segment, sr)

            # Record the extracted file path and the native state.
            records.append({
                "Extracted_File_Path": output_file,
                "Native_State": native_state
            }) # Blank line for clarity between files.

    # Create a new DataFrame from the records and save it as a CSV file.
    db_df = pd.DataFrame(records)
    db_df.to_csv(output_db_csv, index=False)
    print(f"Database saved to {output_db_csv}")

In [53]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd

In [54]:
df_final['File_Path'][4500]


'augmented_audio/Hin_0070_Eng_m_7067_aug2.wav'

In [55]:
df_final.head()

,File_Path,Speaker_ID,Native_State,Duration
0,../Merged_Audio/Hin_0014_Hin_m_0035.wav,Hin_0014,0,7.00000
1,../Merged_Audio/Hin_0052_Eng_m_0001.wav,Hin_0052,0,6.62500
2,../Merged_Audio/Hin_0052_Eng_m_0003.wav,Hin_0052,0,12.09375
3,../Merged_Audio/Hin_0029_Hin_m_0032.wav,Hin_0029,0,7.87500
4,../Merged_Audio/Hin_0052_Eng_m_6490.wav,Hin_0052,0,7.03125


In [56]:
# Assuming df is a Pandas DataFrame
train_df, val_df = train_test_split(df_final, test_size=0.2, random_state=42)

# Convert to DatasetDict
ds = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df),
        "validation": Dataset.from_pandas(val_df),
    }
)

In [57]:
model_name = "mispeech/dasheng-base"

In [58]:
from dasheng_model.feature_extraction_dasheng import DashengFeatureExtractor

feature_extractor = DashengFeatureExtractor.from_pretrained(model_name)

In [59]:
from dasheng_model.feature_extraction_dasheng import DashengFeatureExtractor

def preprocess_function(example):
    max_duration = 3
    target_sr = 16000
    desired_length = int(max_duration * target_sr)  # Target number of samples

    # Load and resample the audio
    audio, orig_sr = librosa.load(example['File_Path'], sr=48000)
    audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=target_sr)

    # If the audio is shorter than the desired length, repeat (tile) it until it reaches the target length.
    if len(audio) < desired_length:
        # Calculate how many repeats are neede-d.
        num_repeats = int(np.ceil(desired_length / len(audio)))
        audio = np.tile(audio, num_repeats)[:desired_length]
    else:
        # Otherwise, truncate the audio to the desired length.
        audio = audio[:desired_length]

    # Use the feature extractor to process the audio
    inputs = feature_extractor(
        audio,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=desired_length,
        truncation=True,
    )

    # Squeeze extra dimensions if needed
    inputs["input_values"] = inputs["input_values"].squeeze(0)
    return {**inputs, "labels": example['Native_State']}

In [60]:
import librosa
import numpy as np

In [61]:
encoded_dataset = ds.map(
   preprocess_function,
    remove_columns=['File_Path','Speaker_ID', 'Native_State','Duration'],
    batched=False,
    with_indices=False,
    with_rank=False,
)

Map:   0%|          | 0/5168 [00:00<?, ? examples/s]

Map:   0%|          | 0/1292 [00:00<?, ? examples/s]

In [62]:
encoded_dataset["validation"][176]

{'__index_level_0__': 1221,
 'input_values': [[-17.727296829223633,
   -18.968612670898438,
   -22.16845703125,
   -22.343809127807617,
   -22.53319549560547,
   -25.059925079345703,
   -28.751644134521484,
   -25.80428123474121,
   -24.003095626831055,
   -21.020442962646484,
   -22.844694137573242,
   -25.58538818359375,
   -22.592872619628906,
   -21.75666046142578,
   -21.20868682861328,
   -19.85591697692871,
   -19.31880760192871,
   -19.64505958557129,
   -18.902671813964844,
   -17.387142181396484,
   -16.949893951416016,
   -18.857990264892578,
   -20.502704620361328,
   -21.714475631713867,
   -24.446765899658203,
   -21.386398315429688,
   -18.649822235107422,
   -19.90791893005371,
   -20.910499572753906,
   -22.367969512939453,
   -29.568077087402344,
   -23.856365203857422,
   -20.666181564331055,
   -22.76740074157715,
   -24.64592170715332,
   -24.297792434692383,
   -26.29173469543457,
   -26.88756561279297,
   -24.16326141357422,
   -24.266033172607422,
   -25.2430152

In [63]:
from transformers import TrainingArguments, Trainer

In [64]:
    from dasheng_model.modeling_dasheng import DashengModel

    outputdim = 19
    model = DashengModel.from_pretrained(model_name, outputdim=outputdim, ignore_mismatched_sizes=True)

    model.freeze_encoder()
    model.config.loss = "CrossEntropyLoss"

Some weights of DashengModel were not initialized from the model checkpoint at mispeech/dasheng-base and are newly initialized: ['outputlayer.0.bias', 'outputlayer.0.weight', 'outputlayer.1.bias', 'outputlayer.1.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [65]:
batch_size = 32
args = TrainingArguments(
    f"{model_name}-esc50",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-3,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    warmup_ratio=0,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    dataloader_num_workers=2,    # Google Colab suggests setting num_worker=2
    push_to_hub=False,
)

/data/himanshu/new_env/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [66]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions[0], axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [67]:
trainer = Trainer(
    model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,

)

/tmp/ipykernel_161700/1451490098.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [68]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,2.475500,2.486448,0.428793
2,2.396700,2.409114,0.530186
3,2.292100,2.328645,0.570433
4,2.255600,2.319359,0.658669
5,2.235400,2.289207,0.719814
6,2.228300,2.287471,0.710526
7,2.241300,2.274714,0.719040
8,2.201200,2.268605,0.739938
9,2.232500,2.264031,0.729102
10,2.210500,2.265910,0.736068


TrainOutput(global_step=1620, training_loss=2.2990308997071818, metrics={'train_runtime': 346.5946, 'train_samples_per_second': 149.108, 'train_steps_per_second': 4.674, 'total_flos': 5.105087752375296e+17, 'train_loss': 2.2990308997071818, 'epoch': 10.0})